In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config

Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
from pyspark.sql import functions as F

# Load both Bronze tables
df_pre = spark.table(f"{BRONZE_PATH}.google_trends_raw")
df_post = spark.table(f"{BRONZE_PATH}.google_trends_postrelease_raw")

# Aggregate using AVERAGE not max
df_pre_agg = (df_pre
    .groupBy("film", "release_date", "genre")
    .agg(
        F.avg("interest_score").alias("pre_release_rating")
    )
)

df_post_agg = (df_post
    .groupBy("film", "release_date", "genre")
    .agg(
        F.avg("interest_score").alias("post_release_rating")
    )
)

# Join on film
df_combined = (df_pre_agg
    .join(df_post_agg.select("film", "post_release_rating"),
          on="film", how="left")
)

# Add release year and era
df_combined = (df_combined
    .withColumn("release_year",
        F.year(F.col("release_date").cast("date")))
    .withColumn("era",
        F.when(F.col("release_year") <= 2014,
            "Pre Short-Form (2005-2014)")
         .otherwise("Short-Form Era (2015-2025)"))
)

# Calculate hype drop and retention
df_combined = (df_combined
    .withColumn("hype_drop",
        (F.col("pre_release_rating") - F.col("post_release_rating"))
         .cast("double"))
    .withColumn("retention_pct",
        F.when(F.col("pre_release_rating") > 0,
            (F.col("post_release_rating") /
             F.col("pre_release_rating") * 100).cast("double"))
         .otherwise(None))
)

# Final selection
df_final = df_combined.select(
    "film",
    "release_year",
    "genre",
    "era",
    "pre_release_rating",
    "post_release_rating",
    "hype_drop",
    "retention_pct"
).orderBy("release_year")

print(f"✅ Combined trends ready — {df_final.count()} films")
display(df_final)

✅ Combined trends ready — 50 films


film,release_year,genre,era,pre_release_rating,post_release_rating,hype_drop,retention_pct
Batman Begins,2005,Action,Pre Short-Form (2005-2014),11.846153846153847,15.912087912087912,-4.065934065934066,134.32282003710574
Brokeback Mountain,2005,Drama,Pre Short-Form (2005-2014),12.87912087912088,45.142857142857146,-32.26373626373626,350.51194539249144
Casino Royale,2006,Action,Pre Short-Form (2005-2014),9.516483516483516,14.791208791208792,-5.2747252747252755,155.42725173210164
The Departed,2006,Drama,Pre Short-Form (2005-2014),6.450549450549451,19.483516483516482,-13.032967032967031,302.04429301533213
Ratatouille,2007,Comedy,Pre Short-Form (2005-2014),8.758241758241759,17.9010989010989,-9.142857142857142,204.39146800501882
No Country for Old Men,2007,Thriller,Pre Short-Form (2005-2014),5.637362637362638,37.67032967032967,-32.032967032967036,668.2261208576998
There Will Be Blood,2007,Drama,Pre Short-Form (2005-2014),9.736263736263735,33.62637362637363,-23.89010989010989,345.37246049661405
The Dark Knight,2008,Action,Pre Short-Form (2005-2014),9.010989010989011,12.813186813186814,-3.802197802197803,142.19512195121953
Paranormal Activity,2009,Horror,Pre Short-Form (2005-2014),11.021978021978022,17.439560439560438,-6.417582417582416,158.22532402791626
The Hangover,2009,Comedy,Pre Short-Form (2005-2014),7.0989010989010985,25.36263736263736,-18.263736263736263,357.2755417956656


In [0]:
(df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.trends_combined")
)

print(f"✅ Saved to {SILVER_PATH}.trends_combined")
print(f"   Rows written: {df_final.count()}")
display(spark.table(f"{SILVER_PATH}.trends_combined"))

✅ Saved to bootcamp_students.tiffani_ceresrain_silver.trends_combined
   Rows written: 50


film,release_year,genre,era,pre_release_rating,post_release_rating,hype_drop,retention_pct
Batman Begins,2005,Action,Pre Short-Form (2005-2014),11.846153846153847,15.912087912087912,-4.065934065934066,134.32282003710574
Brokeback Mountain,2005,Drama,Pre Short-Form (2005-2014),12.87912087912088,45.142857142857146,-32.26373626373626,350.51194539249144
Casino Royale,2006,Action,Pre Short-Form (2005-2014),9.516483516483516,14.791208791208792,-5.2747252747252755,155.42725173210164
The Departed,2006,Drama,Pre Short-Form (2005-2014),6.450549450549451,19.483516483516482,-13.032967032967031,302.04429301533213
Ratatouille,2007,Comedy,Pre Short-Form (2005-2014),8.758241758241759,17.9010989010989,-9.142857142857142,204.39146800501882
No Country for Old Men,2007,Thriller,Pre Short-Form (2005-2014),5.637362637362638,37.67032967032967,-32.032967032967036,668.2261208576998
There Will Be Blood,2007,Drama,Pre Short-Form (2005-2014),9.736263736263735,33.62637362637363,-23.89010989010989,345.37246049661405
The Dark Knight,2008,Action,Pre Short-Form (2005-2014),9.010989010989011,12.813186813186814,-3.802197802197803,142.19512195121953
Paranormal Activity,2009,Horror,Pre Short-Form (2005-2014),11.021978021978022,17.439560439560438,-6.417582417582416,158.22532402791626
The Hangover,2009,Comedy,Pre Short-Form (2005-2014),7.0989010989010985,25.36263736263736,-18.263736263736263,357.2755417956656


In [0]:
(df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.trends_combined")
)

print(f"✅ Saved to {SILVER_PATH}.trends_combined")
print(f"   Rows written: {df_final.count()}")

display(spark.table(f"{SILVER_PATH}.trends_combined"))

✅ Saved to bootcamp_students.tiffani_ceresrain_silver.trends_combined
   Rows written: 50


film,release_year,genre,era,pre_release_rating,post_release_rating,hype_drop,retention_pct
Batman Begins,2005,Action,Pre Short-Form (2005-2014),11.846153846153847,15.912087912087912,-4.065934065934066,134.32282003710574
Brokeback Mountain,2005,Drama,Pre Short-Form (2005-2014),12.87912087912088,45.142857142857146,-32.26373626373626,350.51194539249144
Casino Royale,2006,Action,Pre Short-Form (2005-2014),9.516483516483516,14.791208791208792,-5.2747252747252755,155.42725173210164
The Departed,2006,Drama,Pre Short-Form (2005-2014),6.450549450549451,19.483516483516482,-13.032967032967031,302.04429301533213
Ratatouille,2007,Comedy,Pre Short-Form (2005-2014),8.758241758241759,17.9010989010989,-9.142857142857142,204.39146800501882
No Country for Old Men,2007,Thriller,Pre Short-Form (2005-2014),5.637362637362638,37.67032967032967,-32.032967032967036,668.2261208576998
There Will Be Blood,2007,Drama,Pre Short-Form (2005-2014),9.736263736263735,33.62637362637363,-23.89010989010989,345.37246049661405
The Dark Knight,2008,Action,Pre Short-Form (2005-2014),9.010989010989011,12.813186813186814,-3.802197802197803,142.19512195121953
Paranormal Activity,2009,Horror,Pre Short-Form (2005-2014),11.021978021978022,17.439560439560438,-6.417582417582416,158.22532402791626
The Hangover,2009,Comedy,Pre Short-Form (2005-2014),7.0989010989010985,25.36263736263736,-18.263736263736263,357.2755417956656
